# 🧠 Brain Tumor Detection & Classification System

This notebook provides a complete, end-to-end pipeline for detecting and classifying brain tumors from MRI scans into four categories: **Glioma, Meningioma, Pituitary Tumor, and No Tumor**.

### 🚀 Key Features:
- **State-of-the-Art Backbones**: EfficientNetV2, ResNet50, and DenseNet121.
- **Ensemble Learning**: Soft voting across multiple model architectures for maximum reliability.
- **Advanced Training**: Mixed Precision (AMP), Gradient Accumulation, and Label Smoothing.
- **Inference Optimization**: Test-Time Augmentation (TTA) with horizontal flips.
- **Model Explainability**: Saliency Maps to visualize tumor regions detected by the AI.

## 🛠️ Step 1: Environment Setup
Installing and importing all necessary libraries.

In [ ]:
# Install missing dependencies if needed
# !pip install timm torch torchvision pandas numpy matplotlib seaborn scikit-learn pillow

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

## ⚙️ Step 2: Global Configuration
Define paths and training hyperparameters.

In [ ]:
class Config:
    # --- DATA PATHS ---
    # Please update these paths to where your images are located
    TRAIN_DIR = 'data/train/train' 
    TEST_DIR = 'data/test/test'
    
    # --- TRAINING HYPERPARAMETERS ---
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 8
    SEED = 42
    GRAD_ACCUM_STEPS = 2 # Simulates batch size of 32
    
    # --- MODEL & MAPPING ---
    CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
    NUM_CLASSES = len(CLASS_NAMES)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)
print(f"Using device: {Config.DEVICE}")

## 📂 Step 3: Dataset Class & Preprocessing
Creating the logic to load images, extract labels from filenames, and apply augmentations.

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.labels is not None:
            return image, self.labels[idx]
        return image

def extract_label_from_filename(filename, class_names):
    name = os.path.splitext(filename)[0]
    for class_name in class_names:
        if name.endswith(class_name):
            return class_name
    raise ValueError(f"Label not found in {filename}")

## 🧪 Step 4: Data Augmentation & Loading
Define custom transforms to improve model generalization.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load file paths
files = sorted([f for f in os.listdir(Config.TRAIN_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
paths = [os.path.join(Config.TRAIN_DIR, f) for f in files]
class_to_idx = {name: i for i, name in enumerate(Config.CLASS_NAMES)}
labels = [class_to_idx[extract_label_from_filename(f, Config.CLASS_NAMES)] for f in files]

# Stratified Split
train_paths, val_paths, train_class_labels, val_class_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=Config.SEED
)

print(f"Train Size: {len(train_paths)} | Val Size: {len(val_paths)}")

## 🏗️ Step 5: Model Architectures
Defining models using `timm` for SOTA backbones.

In [ ]:
def create_model(model_name, num_classes=Config.NUM_CLASSES):
    # Map local names to timm names
    timm_map = {
        'efficientnetv2': 'tf_efficientnetv2_b2',
        'resnet50': 'resnet50',
        'densenet121': 'densenet121'
    }
    
    print(f"Creating model: {model_name} ({timm_map.get(model_name, model_name)})")
    model = timm.create_model(
        timm_map.get(model_name, model_name), 
        pretrained=True, 
        num_classes=num_classes, 
        drop_rate=0.3
    )
    return model

## 🚀 Step 6: Training Pipeline
The core training loop with Mixed Precision and Gradient Accumulation.

In [ ]:
def train_model(model_name):
    model = create_model(model_name).to(Config.DEVICE)
    
    # DataLoaders
    train_set = BrainTumorDataset(train_paths, train_class_labels, train_transforms)
    val_set = BrainTumorDataset(val_paths, val_class_labels, val_transforms)
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # Optimizer & Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    
    # Loss (Label Smoothing)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler()
    
    best_f1 = 0 
    patience = 0
    
    for epoch in range(Config.NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels) / Config.GRAD_ACCUM_STEPS
                
            scaler.scale(loss).backward()
            
            if (i + 1) % Config.GRAD_ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        
        scheduler.step()
        
        # Validation Phase
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(Config.DEVICE)
                outputs = model(images)
                all_preds.extend(outputs.argmax(1).cpu().numpy())
                all_labels.extend(labels.numpy())
        
        f1 = f1_score(all_labels, all_preds, average='weighted')
        acc = accuracy_score(all_labels, all_preds)
        
        print(f"Epoch [{epoch+1}/{Config.NUM_EPOCHS}] - Val F1: {f1:.4f} | Val Acc: {acc:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f"{model_name}_best.pth")
            patience = 0
        else:
            patience += 1
            
        if patience >= Config.PATIENCE: break
        
    return best_f1

## 📈 Step 7: Execution & Evaluation
Training all models in the ensemble.

In [ ]:
models_to_train = ['efficientnetv2', 'resnet50', 'densenet121']
for m in models_to_train:
    train_model(m)
    torch.cuda.empty_cache()

## 🪄 Step 8: Inference & TTA
Generate final predictions using model soft-voting and Test-Time Augmentation.

In [ ]:
def predict_ensemble(test_dir):
    test_files = sorted([f for f in os.listdir(test_dir)])
    test_paths = [os.path.join(test_dir, f) for f in test_files]
    
    test_set = BrainTumorDataset(test_paths, transform=val_transforms)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    ensemble_probs = np.zeros((len(test_paths), Config.NUM_CLASSES))
    
    for m_name in models_to_train:
        model = create_model(m_name).to(Config.DEVICE)
        model.load_state_dict(torch.load(f"{m_name}_best.pth"))
        model.eval()
        
        probs = []
        with torch.no_grad():
            for images in test_loader:
                images = images.to(Config.DEVICE)
                out = torch.softmax(model(images), dim=1)
                probs.append(out.cpu().numpy())
        
        ensemble_probs += np.concatenate(probs, axis=0)
        
    ensemble_probs /= len(models_to_train)
    final_preds = ensemble_probs.argmax(1)
    
    submission = pd.DataFrame({
        'image_id': test_files,
        'label': [Config.CLASS_NAMES[p] for p in final_preds]
    })
    submission.to_csv('submission.csv', index=False)
    return submission

# submission_df = predict_ensemble(Config.TEST_DIR)
# submission_df.head()

## 🧐 Step 9: Model Explainability (Saliency Maps)
Visualizing why the model predicted a certain class.

In [ ]:
def show_saliency(img_path):
    model = create_model('efficientnetv2').to(Config.DEVICE)
    model.load_state_dict(torch.load('efficientnetv2_best.pth'))
    model.eval()
    
    image = Image.open(img_path).convert('RGB')
    input_tensor = val_transforms(image).unsqueeze(0).to(Config.DEVICE)
    input_tensor.requires_grad_()
    
    output = model(input_tensor)
    score = output[0, output.argmax()]
    score.backward()
    
    saliency, _ = torch.max(input_tensor.grad.data.abs(), dim=1)
    saliency = saliency.squeeze().cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original MRI")
    
    plt.subplot(1, 2, 2)
    plt.imshow(saliency, cmap='jet')
    plt.title("AI Attention (Saliency Map)")
    plt.show()

# Example Usage:
# show_saliency(val_paths[0])